# 01 — Model setup: iDT1294 for PHB optimization

This initial notebook loads the genome-scale metabolic model iDT1294 of Rhodopseudomonas palustris, developed by Tec-Campos et al. (2023), from .mat to .xml format using the COBRApy framework.

## *Installation

In [ ]:
# Install cobrapy
!pip install -qq cobra escher networkx memote catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.0/961.0 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.8/141.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.2/342.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 

In [ ]:
# Check installation
!pip show cobra

Name: cobra
Version: 0.30.0
Summary: COBRApy is a package for constraint-based modeling of metabolic networks.
Home-page: https://opencobra.github.io/cobrapy
Author: The cobrapy core development team.
Author-email: cobra-pie@googlegroups.com
License: LGPL-2.0-or-later OR GPL-2.0-or-later
Location: /usr/local/lib/python3.12/dist-packages
Requires: appdirs, depinfo, diskcache, future, httpx, numpy, optlang, pandas, pydantic, python-libsbml, rich, ruamel.yaml, swiglpk
Required-by: Escher, memote


## *Mount drive

In [ ]:
# Mount drive

from google.colab import drive

drive_folder = "metabolic_modelling/phb-optimization-rpalustris/"
def mount_drive():
  drive.mount('/content/drive', force_remount=True)
  os.chdir('/content/drive/MyDrive/'+ drive_folder)
mount_drive()

# Project root = current working directory
project_root = Path(os.getcwd())

Mounted at /content/driveDL


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/metabolic_modelling/phb-optimization-rpalustris/')
RAW_DATA_DIR = DRIVE_ROOT / 'data'
INTERMEDIATE_DATA_DIR = DRIVE_ROOT / 'intermediate_data'
FIGURES_DIR = DRIVE_ROOT / 'results'



## *Load packages

In [ ]:
# load packages

import logging
from pathlib import Path

import cobra
from cobra import Reaction, Metabolite, Model, io
from cobra.io import (
    load_model, load_json_model, save_json_model,
    load_matlab_model, save_matlab_model,
    read_sbml_model, write_sbml_model
)
from cobra.util.solver import linear_reaction_coefficients

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import seaborn as sns
import json
import os
import math
import copy
from itertools import product
from tqdm import tqdm


In [ ]:
!python --version

Python 3.12.12


## *Load model

In [ ]:
# Project root = current working directory
project_root = Path(os.getcwd())
print(project_root)


/content/driveDL/My Drive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/models


In [ ]:
# Load models

# Path to base model
base_model_dir = project_root / "base"
base_model_dir.mkdir(parents=True, exist_ok=True)

# Full path to the SBML file
#base_model_path = base_model_dir / "model_rpalustris_comp_fixed.xml" # this model was opened and converted from matlab to sbml (compartments data may change)

# Load model
#model_base = io.read_sbml_model(
#    str(base_model_path),
#    use_fbc=False
#)

project_root_palustris = Path(
    "/content/driveDL/MyDrive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/"
    "rpalustris_pha_optimization/models/matlab"
)

rp_model_mat_path = project_root_palustris / "iDT1294.mat"

model_base = load_matlab_model(str(rp_model_mat_path))

print("Model loaded successfully")

Model loaded successfully


# Check default model tec_campos_iDT1294

In [ ]:
# Load model
model_base

Name,iDT1294
Memory address,7b499cfc3350
Number of metabolites,2124
Number of reactions,2721
Number of genes,1294
Number of groups,118
Objective expression,1.0*BIOMASS__1 - 1.0*BIOMASS__1_reverse_063c7
Compartments,"c, u, p, e"


In [ ]:
# Check model summary
model_base.summary() # BIOMASS__1 is the default objective function

Metabolite,Reaction,Flux,C-Number,C-Flux
actn__R_e,EX_actn__R_e,0.6784,4,12.28%
ca2_e,EX_ca2_e,1.03E-05,0,0.00%
cinnm_e,EX_cinnm_e,2.153,9,87.72%
cl_e,EX_cl_e,2.06E-05,0,0.00%
cobalt2_e,EX_cobalt2_e,4.533E-06,0,0.00%
fe3_e,EX_fe3_e,0.0337,0,0.00%
k_e,EX_k_e,1.366E-06,0,0.00%
mg2_e,EX_mg2_e,0.01097,0,0.00%
mn2_e,EX_mn2_e,1.554E-07,0,0.00%
na1_e,EX_na1_e,1.051E-07,0,0.00%


In [ ]:
# Check model compartments
model_base.compartments

{'c': 'c', 'u': 'u', 'p': 'p', 'e': 'e'}

## PHB reactions

In [ ]:
# Check reaction directly associated to PHB production
phb_rxn = model_base.reactions.get_by_id("PHBS_syn")
print(phb_rxn.bounds)
print(phb_rxn.reaction)


(0.0, 999999.0)
3hbcoa__R_c + phbg_c --> PHB_c + coa_c


# *Adjust constraints

In [ ]:
# Adjust constraints since some reactions have values ><1000
for rxn in model_base.reactions:
    if rxn.lower_bound < -1000:
        rxn.lower_bound = -1000
    if rxn.upper_bound > 1000:
        rxn.upper_bound = 1000

# *Save base model (PHB)

In [ ]:
# Save model with phbv reactions in phbv directory
phb_model_dir = project_root / "phb"

# Ensure folder exists
phb_model_dir.mkdir(parents=True, exist_ok=True)

# Target file
phb_model_file = phb_model_dir / "model_rpalustris_PHB.xml"

# Save COBRA model as .xml file
cobra.io.write_sbml_model(model_base, str(phb_model_file))
